# Network Visualization Tool for Essays and Persons

This notebook provides an easy way to visualize relationships between essays and the persons mentioned in them.

## What you'll need:
- Your CSV files with essay data (named like `essays 01_01 persons.csv`)
- Python 3.8 or higher

## Quick Start:
1. Run the setup cell below
2. Run the visualization cell with your data path
3. Open http://localhost:8000 in your browser

## 1. Setup (Run this first!)

In [ ]:
# Install required packages
import subprocess
import sys

def install_package(package):
    try:
        __import__(package)
        print(f"✅ {package} already installed")
    except ImportError:
        print(f"📦 Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ {package} installed")

# Install required packages
packages = ['pandas', 'requests', 'flask', 'numpy']
for package in packages:
    install_package(package)

print("\n🎉 Setup complete! You're ready to visualize your data.")

## 2. Import Libraries

In [ ]:
import pandas as pd
import requests
import json
import subprocess
import sys
import os
import threading
import time
from pathlib import Path
from urllib.parse import urlparse
import re

print("✅ Libraries imported successfully!")

## 3. Visualization Functions

In [ ]:
def start_server():
    """Start the Flask server in the background"""
    try:
        # Create a simple Flask app
        flask_code = '''
from flask import Flask, request, jsonify, send_from_directory
from flask_cors import CORS
import json
import os

app = Flask(__name__)
CORS(app)

# Store the current data
current_data = {"nodes": [], "edges": []}

@app.route('/')
def index():
    return '''
<!DOCTYPE html>
<html>
<head>
    <title>Network Visualization</title>
    <script src="https://unpkg.com/three@0.157.0/build/three.min.js"></script>
    <script src="https://unpkg.com/three@0.157.0/examples/js/controls/OrbitControls.js"></script>
    <style>
        body { margin: 0; font-family: Arial, sans-serif; }
        #container { width: 100vw; height: 100vh; }
        #info { position: absolute; top: 10px; left: 10px; background: rgba(0,0,0,0.7); color: white; padding: 10px; border-radius: 5px; }
        #controls { position: absolute; top: 10px; right: 10px; background: rgba(0,0,0,0.7); color: white; padding: 10px; border-radius: 5px; }
    </style>
</head>
<body>
    <div id="container"></div>
    <div id="info">Loading...</div>
    <div id="controls">
        <h3>Controls</h3>
        <p>Mouse: Rotate</p>
        <p>Scroll: Zoom</p>
        <p>Right-click: Pan</p>
    </div>
    <script>
        // Initialize Three.js
        const scene = new THREE.Scene();
        const camera = new THREE.PerspectiveCamera(75, window.innerWidth / window.innerHeight, 0.1, 1000);
        const renderer = new THREE.WebGLRenderer();
        renderer.setSize(window.innerWidth, window.innerHeight);
        document.getElementById('container').appendChild(renderer.domElement);

        // Add controls
        const controls = new THREE.OrbitControls(camera, renderer.domElement);
        controls.enableDamping = true;
        controls.dampingFactor = 0.05;

        // Add lighting
        const ambientLight = new THREE.AmbientLight(0x404040);
        scene.add(ambientLight);
        const directionalLight = new THREE.DirectionalLight(0xffffff, 0.5);
        directionalLight.position.set(1, 1, 1);
        scene.add(directionalLight);

        // Load data
        fetch('/api/data')
            .then(response => response.json())
            .then(data => {
                const nodes = data.nodes || [];
                const edges = data.edges || [];
                
                // Create nodes
                const nodeGeometry = new THREE.SphereGeometry(0.1);
                const nodeMaterial = new THREE.MeshBasicMaterial({ color: 0x00ff00 });
                
                nodes.forEach((node, index) => {
                    const sphere = new THREE.Mesh(nodeGeometry, nodeMaterial);
                    sphere.position.set(
                        (Math.random() - 0.5) * 10,
                        (Math.random() - 0.5) * 10,
                        (Math.random() - 0.5) * 10
                    );
                    sphere.userData = node;
                    scene.add(sphere);
                });
                
                // Create edges
                edges.forEach(edge => {
                    const sourceNode = nodes.find(n => n.id === edge.source);
                    const targetNode = nodes.find(n => n.id === edge.target);
                    if (sourceNode && targetNode) {
                        const geometry = new THREE.BufferGeometry().setFromPoints([
                            new THREE.Vector3(sourceNode.x || 0, sourceNode.y || 0, sourceNode.z || 0),
                            new THREE.Vector3(targetNode.x || 0, targetNode.y || 0, targetNode.z || 0)
                        ]);
                        const material = new THREE.LineBasicMaterial({ color: 0x888888 });
                        const line = new THREE.Line(geometry, material);
                        scene.add(line);
                    }
                });
                
                // Update info
                document.getElementById('info').innerHTML = 
                    `<h3>Network Visualization</h3><p>${nodes.length} nodes, ${edges.length} edges</p>`;
            })
            .catch(error => {
                console.error('Error loading data:', error);
                document.getElementById('info').innerHTML = '<h3>Error loading data</h3>';
            });

        // Position camera
        camera.position.z = 5;

        // Animation loop
        function animate() {
            requestAnimationFrame(animate);
            controls.update();
            renderer.render(scene, camera);
        }
        animate();

        // Handle window resize
        window.addEventListener('resize', onWindowResize, false);
        function onWindowResize() {
            camera.aspect = window.innerWidth / window.innerHeight;
            camera.updateProjectionMatrix();
            renderer.setSize(window.innerWidth, window.innerHeight);
        }
    </script>
</body>
</html>
        '''

@app.route('/api/data')
def get_data():
    return jsonify(current_data)

@app.route('/api/data', methods=['POST'])
def update_data():
    global current_data
    current_data = request.json
    return jsonify({"status": "success"})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=8000, debug=False)
'''
        
        # Write Flask app to file
        with open('temp_server.py', 'w') as f:
            f.write(flask_code)
        
        # Start server in background
        def run_server():
            subprocess.run([sys.executable, 'temp_server.py'], 
                          stdout=subprocess.DEVNULL, 
                          stderr=subprocess.DEVNULL)
        
        server_thread = threading.Thread(target=run_server, daemon=True)
        server_thread.start()
        
        # Wait a moment for server to start
        time.sleep(2)
        
        print("✅ Server started at http://localhost:8000")
        return True
        
    except Exception as e:
        print(f"❌ Failed to start server: {e}")
        return False

def visualize_data(data_path, server_url="http://localhost:8000"):
    """Visualize data from a folder containing CSV files"""
    
    print(f"📊 Loading data from: {data_path}")
    
    # Find all CSV files
    csv_files = []
    data_path = Path(data_path)
    
    if data_path.is_file():
        csv_files = [data_path]
    elif data_path.is_dir():
        # Look for CSV files in the directory and subdirectories
        for pattern in ['*.csv', '**/*.csv']:
            csv_files.extend(data_path.glob(pattern))
    
    if not csv_files:
        print(f"❌ No CSV files found in {data_path}")
        return False
    
    print(f"✅ Found {len(csv_files)} CSV files")
    
    # Load all data
    all_nodes = []
    all_edges = []
    node_id_counter = 1
    
    for csv_file in csv_files:
        try:
            # Try different encodings
            for encoding in ['utf-8', 'latin-1', 'cp1252']:
                try:
                    df = pd.read_csv(csv_file, encoding=encoding)
                    break
                except UnicodeDecodeError:
                    continue
            else:
                print(f"❌ Error loading {csv_file}: encoding issues")
                continue
            
            # Extract person names
            if 'person name' in df.columns:
                persons = df['person name'].dropna().astype(str).tolist()
            elif 'name' in df.columns:
                persons = df['name'].dropna().astype(str).tolist()
            else:
                persons = df.iloc[:, 0].dropna().astype(str).tolist()
            
            # Filter out empty strings and 'nan'
            persons = [p for p in persons if p and p.lower() != 'nan']
            
            print(f"✅ Loaded {len(persons)} persons from {csv_file.name}")
            
            # Create essay node
            essay_name = csv_file.stem
            essay_node = {
                "id": f"essay_{csv_file.name}",
                "label": essay_name,
                "type": "essay",
                "x": (len(all_nodes) % 10 - 5) * 2,
                "y": (len(all_nodes) // 10 - 5) * 2,
                "z": 0
            }
            all_nodes.append(essay_node)
            
            # Create person nodes and connections
            for person in persons:
                person_id = f"person_{person}_{csv_file.name}"
                person_node = {
                    "id": person_id,
                    "label": person,
                    "type": "person",
                    "x": (len(all_nodes) % 20 - 10) * 0.5,
                    "y": (len(all_nodes) // 20 - 10) * 0.5,
                    "z": (len(all_nodes) % 5 - 2) * 0.5
                }
                all_nodes.append(person_node)
                
                # Create connection
                all_edges.append({
                    "source": essay_node["id"],
                    "target": person_id,
                    "label": "mentions"
                })
                
        except Exception as e:
            print(f"❌ Error loading {csv_file}: {e}")
            continue
    
    # Prepare data for frontend
    data = {
        "nodes": all_nodes,
        "edges": all_edges,
        "metadata": {
            "total_nodes": len(all_nodes),
            "total_edges": len(all_edges),
            "files_processed": len(csv_files)
        }
    }
    
    # Send to frontend
    print("🌐 Sending to frontend...")
    try:
        response = requests.post(
            f"{server_url}/api/data",
            json=data,
            headers={'Content-Type': 'application/json'}
        )
        
        if response.status_code == 200:
            print("✅ Data visualized successfully!")
            print(f"📱 Open {server_url} to see your visualization")
            print(f"📊 {len(all_nodes)} nodes, {len(all_edges)} edges")
            return True
        else:
            print(f"❌ Failed to send data: {response.status_code}")
            return False
            
    except Exception as e:
        print(f"❌ Error sending data: {e}")
        return False

print("✅ Visualization functions loaded!")

## 4. Start the Server

In [ ]:
# Start the visualization server
if start_server():
    print("\n🎉 Server is running! You can now visualize your data.")
    print("📱 Open http://localhost:8000 in your browser to see the interface.")
else:
    print("\n❌ Failed to start server. Please check the error messages above.")

## 5. Visualize Your Data

Run the cell below with your data path:

In [ ]:
# Replace this path with your data folder or CSV file
data_path = "/path/to/your/essays/folder"  # Change this!

# Examples:
# data_path = "/Users/student/Downloads"
# data_path = "/Users/student/Programming Historian lesson"
# data_path = "essays 02_05 persons.csv"  # Single file

# Visualize the data
visualize_data(data_path)

## 6. Try with Sample Data (Optional)

If you don't have your own data yet, you can try with sample data:

In [ ]:
# Create sample data
import tempfile
import os

# Create a temporary directory with sample CSV files
with tempfile.TemporaryDirectory() as temp_dir:
    # Sample essay 1
    essay1_content = """person name,count
Aristotle,5
Plato,3
Socrates,2
Cicero,4"""
    
    with open(os.path.join(temp_dir, "essays 01_01 persons.csv"), 'w') as f:
        f.write(essay1_content)
    
    # Sample essay 2
    essay2_content = """person name,count
Caesar,3
Augustus,2
Cicero,1
Virgil,2"""
    
    with open(os.path.join(temp_dir, "essays 01_02 persons.csv"), 'w') as f:
        f.write(essay2_content)
    
    print(f"📝 Created sample data in: {temp_dir}")
    
    # Visualize the sample data
    visualize_data(temp_dir)

## 7. Troubleshooting

If something goes wrong:

### Port already in use:
```python
import os
os.system("lsof -ti:8000 | xargs kill -9")
```

### Check if server is running:
```python
import requests
try:
    response = requests.get("http://localhost:8000/api/data")
    print("✅ Server is running")
except:
    print("❌ Server is not running")
```

### Restart the server:
```python
start_server()
```

## 🎉 You're Done!

1. **Open http://localhost:8000** in your browser
2. **Explore your network** - rotate, zoom, pan
3. **Look for patterns** - which essays share the same persons?
4. **Analyze connections** - who appears in multiple essays?

### Tips:
- **Green nodes** = Essays
- **Gray lines** = Connections between essays and persons
- **Mouse controls**: Rotate, scroll to zoom, right-click to pan
- **Look for clusters** of connected nodes

Happy visualizing! 🚀